In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Import all imp library

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import wandb
import random, warnings

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

from xgboost import XGBClassifier
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from kaggle_secrets import UserSecretsClient


warnings.filterwarnings('ignore')

seed = 42
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print("Device Name: ", torch.cuda.get_device_name())
print("GPU: ", device)

Device Name:  Tesla T4
GPU:  cuda


# Load Datasets

In [3]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [4]:
train_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [5]:
test_df.head()

,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...


# EDA

In [6]:
print(f'Train shape: {train_df.shape}, Test Shape: {test_df.shape}')
print(f'Train Columns {list(train_df.columns)}')
print(f'Test Columns: {list(test_df.columns)}')
print(f'Train Dtype: {train_df.dtypes}, Test Dtype: {test_df.dtypes}')

Train shape: (2000, 8), Test Shape: (500, 7)
Train Columns ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']
Test Columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E']
Train Dtype: id         int64
prompt    object
A         object
B         object
C         object
D         object
E         object
answer    object
dtype: object, Test Dtype: id         int64
prompt    object
A         object
B         object
C         object
D         object
E         object
dtype: object


In [7]:
print("Null Values")
print(f'Train Null Values: {train_df.isnull().sum()}')
print(f'Test Null Values: {test_df.isnull().sum()}')

Null Values
Train Null Values: id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64
Test Null Values: id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
dtype: int64


In [8]:
# answer distribution
print(train_df.answer.value_counts().sort_index())

answer
A    369
B    490
C    459
D    358
E    324
Name: count, dtype: int64


# Preprocessing

In [9]:
train_df_unique = train_df.drop_duplicates(subset=["prompt"]).copy()

def combine_mcq(row):
    return f"Question: {row['prompt']} Options: A) {row['A']} B) {row['B']} C) {row['C']} D) {row['D']} E) {row['E']}"

train_df_unique['text'] = train_df_unique.apply(combine_mcq, axis=1)
test_df['text'] = test_df.apply(combine_mcq, axis=1)



In [10]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df_unique['text'], train_df_unique['answer'], 
    test_size=0.2, random_state=42, stratify=train_df_unique['answer']
)

print(f"Training samples: {len(train_texts)}")
print(f"Validation samples: {len(val_texts)}")

le = LabelEncoder()
train_labels_enc = le.fit_transform(train_labels)
val_labels_enc = le.transform(val_labels)

Training samples: 1406
Validation samples: 352


In [11]:
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB_API_KEY")

wandb.login(key=wandb_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23f3003059 (23f3003059-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [12]:
# Model 1 Built from scratch
wandb.init(project="23f3003059-t22026", name="MLP_Scratch", config={"metric": "MAP@3"})

tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = tfidf.fit_transform(train_texts)
X_val_tfidf = tfidf.transform(val_texts)
X_train_tensor = torch.FloatTensor(X_train_tfidf.toarray()).to(device)
y_train_tensor = torch.LongTensor(train_labels_enc).to(device)
X_val_tensor = torch.FloatTensor(X_val_tfidf.toarray()).to(device)

class MCQNet(nn.Module):
    def __init__(self, input_dim):
        super(MCQNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 128), nn.ReLU(), nn.Linear(128, 5)
        )
    def forward(self, x): return self.net(x)
        
model_scratch = MCQNet(X_train_tensor.shape[1]).to(device)
optimizer = torch.optim.AdamW(model_scratch.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=32, shuffle=True)
model_scratch.train()

for epoch in range(15):
    for bx, by in train_loader:
        optimizer.zero_grad()
        loss = criterion(model_scratch(bx), by)
        loss.backward()
        optimizer.step()
model_scratch.eval()

with torch.no_grad():
    scratch_val_preds = torch.argmax(model_scratch(X_val_tensor), dim=1).cpu().numpy()
acc_scratch = accuracy_score(val_labels_enc, scratch_val_preds)
f1_scratch = f1_score(val_labels_enc, scratch_val_preds, average='weighted')

wandb.log({"model": "MLP_Scratch", "accuracy": acc_scratch, "f1": f1_scratch})
wandb.finish()



accuracy,▁
f1,▁
accuracy,1
f1,1
model,MLP_Scratch


In [13]:
# Model 2
wandb.init(project="23f3003059-t22026", name="DistilBERT_Pretrained", config={"metric": "MAP@3"})

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
def tokenize_fn(ex):
    return tokenizer(ex["text"], padding="max_length", truncation=True, max_length=512)
    
train_ds = Dataset.from_dict({"text": train_texts, "labels": train_labels_enc}).map(tokenize_fn, batched=True)
val_ds = Dataset.from_dict({"text": val_texts, "labels": val_labels_enc}).map(tokenize_fn, batched=True)
train_ds.set_format("torch"); val_ds.set_format("torch")

model_pretrained = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=5)

args = TrainingArguments(
    output_dir="./res", 
    num_train_epochs=3, 
    per_device_train_batch_size=16, 
    eval_strategy="epoch", 
    save_strategy="epoch", 
    load_best_model_at_end=True, 
    metric_for_best_model="accuracy", 
    report_to="wandb", 
    fp16=True
)


def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    return {
        "accuracy": accuracy_score(p.label_ids, preds),
        "f1": f1_score(p.label_ids, preds, average='weighted')
    }

trainer = Trainer(
    model=model_pretrained,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)
trainer.train()

eval_results = trainer.evaluate()
wandb.log({"model": "DistilBERT_Pretrained", "accuracy": eval_results["eval_accuracy"], "f1": eval_results["eval_f1"]})
wandb.finish()

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1406 [00:00<?, ? examples/s]

Map:   0%|          | 0/352 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


accuracy,▁
eval/accuracy,▁███
eval/f1,▁███
eval/loss,█▂▁▂
eval/runtime,▁▄█▇
eval/samples_per_second,█▅▁▂
eval/steps_per_second,█▅▁▂
f1,▁
train/epoch,▁▅███
train/global_step,▁▅████
accuracy,0.99432


In [14]:
# Model 3 Additional model
wandb.init(project="23f3003059-t22026", name="XGBoost_Additional", config={"metric": "MAP@3"})

xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42)
xgb_model.fit(X_train_tfidf, train_labels_enc)
xgb_val_preds = xgb_model.predict(X_val_tfidf)
acc_xgb = accuracy_score(val_labels_enc, xgb_val_preds)
f1_xgb = f1_score(val_labels_enc, xgb_val_preds, average='weighted')

wandb.log({"model": "XGBoost_Additional", "accuracy": acc_xgb, "f1": f1_xgb})
wandb.finish()

accuracy,▁
f1,▁
accuracy,0.99432
f1,0.99432
model,XGBoost_Additional


In [15]:
# 1. MLP Probs
X_test_tensor = torch.FloatTensor(tfidf.transform(test_df['text']).toarray()).to(device)
model_scratch.eval()
with torch.no_grad():
    mlp_probs = torch.softmax(model_scratch(X_test_tensor), dim=1).cpu().numpy()

# 2. DistilBERT Probs
model_pretrained.eval()
all_bert_probs = []
for i in range(0, len(test_df['text']), 16):
    batch = test_df['text'].iloc[i:i+16].tolist()
    inputs = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.no_grad():
        all_bert_probs.append(torch.nn.functional.softmax(model_pretrained(**inputs).logits, dim=-1).cpu().numpy())
bert_probs = np.vstack(all_bert_probs)

# 3. XGBoost Probs
xgb_probs = xgb_model.predict_proba(tfidf.transform(test_df['text']))
final_probs = (0.4 * bert_probs) + (0.4 * xgb_probs) + (0.2 * mlp_probs)
top_3_indices = np.argsort(final_probs, axis=1)[:, -3:][:, ::-1]
label_map = {i: label for i, label in enumerate(le.classes_)}
predictions = [" ".join([label_map[idx] for idx in indices]) for indices in top_3_indices]
submission = pd.DataFrame({"ID": test_df["id"], "Prediction": predictions})
submission.to_csv("submission.csv", index=False)

In [16]:
pd.read_csv("submission.csv")

,ID,Prediction
0,1,A E D
1,2,B C A
2,3,B C D
3,4,E A D
4,5,C A E
...,...,...
495,496,A D E
496,497,C E B
497,498,B C A
498,499,E D A
